# End-to-End Workflow — calling the platform's own functions

This notebook demonstrates how to execute the **complete data workflow** by
calling the functions in `src/asx_quant` directly (the same functions the
`asx_quant` CLI drives). The full chain:

| step | CLI command | function | what it does |
|---|---|---|---|
| 1. download | `asx_quant ingest` | `stages.run_ingest` | Yahoo Finance + Wikipedia -> Bronze |
| 2. validate | `asx_quant validate` | `stages.run_validate` | data-quality rules on Bronze |
| 3. transform | `asx_quant transform` | `stages.run_transform` | Bronze -> Silver (clean + calendar) |
| 4. features | `asx_quant features` | `stages.run_features` | Silver -> Gold features + labels |
| 5. train | `asx_quant train` | `stages.run_train` | walk-forward training -> OOS predictions |
| 6. portfolio | `asx_quant portfolio` | `stages.run_portfolio` | predictions -> weekly top-20 lists |
| 7. simulate | `asx_quant backtest` | `stages.run_backtest` | exact cost-inclusive simulation |
| 8. report | `asx_quant report` | `stages.run_report_stage` | platform snapshot |

**Two regimes:**

* **Run from the local lake** (this notebook): the steps below execute
  in-process. `ingest --mode incremental` appends only new sessions, and
  every later stage rebuilds its dataset deterministically — a full refresh
  is never needed to stay current.
* **Full refresh from sources**: run
  `uv run asx_quant update --with-sensitivity`, or flip the guarded cell
  below to `RUN_FULL_REFRESH = True`. The only slow stage is *train*; it is
  guarded the same way (set `RUN_SLOW_TRAIN = True` to run it here).

> **Research only — not financial advice.**

In [1]:
import time

import pandas as pd

from asx_quant.config.loader import load_settings
from asx_quant.pipeline import stages
from asx_quant.pipeline.health import run_health_check
from asx_quant.reporting import artifacts

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

settings = load_settings()
print("settings loaded (config/*.yaml) — lake root:", artifacts.lake_root())

settings loaded (config/*.yaml) — lake root: C:\Users\henry\projects\asx-quant-ml\data


### Health check first

The foundation self-probe (storage, config, registry) uses a temporary
sandbox, so it never mutates real data.

In [2]:
health = run_health_check(settings)
print(health.summary())

2026-11-15 13:24:14 | INFO     | asx_quant.health | Health check passed: config: five YAML files load and validate


2026-11-15 13:24:14 | INFO     | asx_quant.health | Health check passed: storage: directory skeleton resolves


2026-11-15 13:24:14 | INFO     | asx_quant.storage.duckdb | DuckDB warehouse initialised at 'C:\Users\henry\AppData\Local\Temp\tmpgvbn2bum\health.duckdb'


2026-11-15 13:24:14 | INFO     | asx_quant.health | Health check passed: storage: DuckDB schemas + metadata initialise


2026-11-15 13:24:14 | INFO     | asx_quant.storage.parquet | Wrote 2 rows to C:\Users\henry\AppData\Local\Temp\tmpp9ipfppx\probe.parquet (probe)


2026-11-15 13:24:14 | INFO     | asx_quant.health | Health check passed: storage: Parquet write/read round-trip


2026-11-15 13:24:14 | INFO     | asx_quant.runs | Run started: 20261115T022414Z-c6a8d366


2026-11-15 13:24:14 | INFO     | asx_quant.runs | Stage started: probe_stage


2026-11-15 13:24:14 | INFO     | asx_quant.runs | Stage probe_stage: success


2026-11-15 13:24:14 | INFO     | asx_quant.health | Health check passed: observability: run registry appends and reads


Health check: HEALTHY
PASS  config: five YAML files load and validate  - project='asx-quant' v0.1.0; duckdb schemas: bronze, gold, meta, silver
PASS  storage: directory skeleton resolves  - bronze domains: _probe, abs, announcements, asic, prices, rba, security_master
PASS  storage: DuckDB schemas + metadata initialise  - schemas: ['bronze', 'gold', 'meta', 'silver']
PASS  storage: Parquet write/read round-trip  - 2 rows; audit columns stamped
PASS  observability: run registry appends and reads  - run 20261115T022414Z-c6a8d366; stage recorded


### Regime 2 — full refresh from sources (guarded)

One call chains every stage and aborts at the first failure. Off by
default: it re-downloads the full history of all 200 tickers and re-trains
every model.

In [3]:
RUN_FULL_REFRESH = False  # set True to re-download everything and retrain
if RUN_FULL_REFRESH:
    stages.run_update_stage(settings, mode="historical", with_sensitivity=True)
else:
    print("skipped — this notebook runs regime 1 (incremental) below instead")

skipped — this notebook runs regime 1 (incremental) below instead


### Regime 1 — the live incremental run

Each stage is called with its real function and timed. `ingest` in
incremental mode appends only sessions newer than the lake (2s when
up to date); `validate` enforces the data-quality rules; `transform` and
`features` rebuild Silver and Gold deterministically.

In [4]:
def timed(name: str, fn, *args, **kwargs):
    started = time.perf_counter()
    summary = fn(*args, **kwargs)
    print(f"[{time.perf_counter() - started:6.1f}s] {name}")
    return summary


ing = timed("ingest (incremental)", stages.run_ingest, settings, "incremental")
val = timed("validate", stages.run_validate, settings)
print(f"   validate: {val.dataset} — status {'PASS' if val.passed else 'FAIL'}, "
      f"{len(val.results)} rules, {val.warn_count()} warnings")
tr = timed("transform", stages.run_transform, settings)
fe = timed("features", stages.run_features, settings)

2026-11-15 13:24:15 | INFO     | asx_quant.runs | Run started: 20261115T022415Z-466cce6f


2026-11-15 13:24:15 | INFO     | asx_quant.runs | Stage started: ingest_universe


2026-11-15 13:24:16 | INFO     | asx_quant.ingestion.universe | Fetched 200 ASX 200 constituents from Wikipedia


2026-11-15 13:24:16 | INFO     | asx_quant.storage.parquet | Wrote 200 rows to C:\Users\henry\projects\asx-quant-ml\data\bronze\security_master\asx200\20261115T022415Z-466cce6f.parquet (security_master_bronze)


2026-11-15 13:24:16 | INFO     | asx_quant.runs | Stage ingest_universe: success


2026-11-15 13:24:16 | INFO     | asx_quant.runs | Stage started: ingest_prices


... (299 outputs left out)

2026-11-15 13:27:38 | INFO     | asx_quant.storage.duckdb | Registered view gold.features (455168 rows) over 'C:\Users\henry\projects\asx-quant-ml\data\gold\features.parquet'


2026-11-15 13:27:38 | INFO     | asx_quant.storage.duckdb | Registered view gold.target_labels (455168 rows) over 'C:\Users\henry\projects\asx-quant-ml\data\gold\target_labels.parquet'


2026-11-15 13:27:39 | INFO     | asx_quant.runs | Stage features: success


[  17.7s] features


### Train (guarded — the only slow stage)

25 walk-forward folds x 4 models over ~455k feature rows. Left off in this
demonstration; the existing out-of-sample predictions are used below.

In [5]:
RUN_SLOW_TRAIN = False  # set True to retrain all models in this session
if RUN_SLOW_TRAIN:
    timed("train", stages.run_train, settings)
else:
    preds = pd.read_parquet("data/gold/predictions.parquet")
    print(f"skipped (slow) — using existing OOS predictions: {len(preds):,} rows, "
          f"{preds['model_name'].nunique()} models, "
          f"{preds['date'].min()} -> {preds['date'].max()}")

skipped (slow) — using existing OOS predictions: 1,135,828 rows, 4 models, 2020-11-06 00:00:00 -> 2026-11-09 00:00:00


### Portfolio and backtest

With fresh features and predictions in place, the portfolio stage turns
them into the weekly top-20 target lists and the backtest stage replays
them through history with slippage, brokerage, whole shares and dividends.

In [6]:
pf = timed("portfolio", stages.run_portfolio, settings)
bt = timed("backtest", stages.run_backtest, settings)

2026-11-15 13:27:39 | INFO     | asx_quant.runs | Run started: 20261115T022739Z-267c876b


2026-11-15 13:27:39 | INFO     | asx_quant.runs | Stage started: portfolio


2026-11-15 13:27:40 | INFO     | asx_quant.portfolio.build | Rebalance schedule: 314 dates, 2020-11-09 -> 2026-11-09


2026-11-15 13:27:49 | INFO     | asx_quant.portfolio.build | Portfolio quality: pass (1 warnings)


2026-11-15 13:27:49 | INFO     | asx_quant.runs | Stage portfolio: success


[   9.7s] portfolio
2026-11-15 13:27:49 | INFO     | asx_quant.runs | Run started: 20261115T022749Z-3443956e


2026-11-15 13:27:49 | INFO     | asx_quant.runs | Stage started: backtest


2026-11-15 13:27:53 | INFO     | asx_quant.backtest.engine | Backtest simulated 1522 sessions, 7571 trades, final NAV 156932.66


2026-11-15 13:27:54 | INFO     | asx_quant.backtest.build | Backtest quality: pass (2 warnings)


2026-11-15 13:27:54 | INFO     | asx_quant.runs | Stage backtest: success


[   5.3s] backtest


### The latest trades

What the strategy would hold at the most recent rebalance, and the last
executed trades in the simulation.

In [7]:
port = artifacts.load_target_portfolios()
latest_date = port["rebalance_date"].max()
latest = port[port["rebalance_date"] == latest_date]
print(f"target portfolio for {latest_date}: {len(latest)} names")
latest[["ticker", "rank", "signal", "target_weight"]].reset_index(drop=True)

target portfolio for 2026-11-09 00:00:00: 20 names


,ticker,rank,signal,target_weight
0,CTD.AX,1,0.024168,0.05
1,DRO.AX,2,0.013187,0.05
2,APE.AX,3,0.011044,0.05
3,DNL.AX,4,0.010583,0.05
4,REG.AX,5,0.010295,0.05
5,TLX.AX,6,0.009330,0.05
6,CDA.AX,7,0.008974,0.05
7,ORI.AX,8,0.008485,0.05
8,LLC.AX,9,0.007977,0.05
9,PXA.AX,10,0.007596,0.05


In [8]:
trades = artifacts.load_trades()
print(f"last {len(trades.tail(8))} of {len(trades):,} simulated trades:")
trades.tail(8).reset_index(drop=True)

last 8 of 7,571 simulated trades:


,effective_date,rebalance_date,ticker,side,shares,price_exec,trade_value,cost_aud,nav_base
0,2026-11-10,2026-11-09,PXA.AX,buy,1044,7.42742,7754.23,13.3771,155216.85
1,2026-11-10,2026-11-09,RDX.AX,buy,2328,3.33333,7759.99,13.3800,155216.85
2,2026-11-10,2026-11-09,REG.AX,buy,1526,5.08508,7759.83,13.3799,155216.85
3,2026-11-10,2026-11-09,SEK.AX,buy,534,14.51450,7750.74,13.3754,155216.85
4,2026-11-10,2026-11-09,SGP.AX,buy,1719,4.51451,7760.44,13.3802,155216.85
5,2026-11-10,2026-11-09,SUL.AX,buy,570,13.59358,7748.34,13.3742,155216.85
6,2026-11-10,2026-11-09,TLX.AX,buy,549,14.11410,7748.64,13.3743,155216.85
7,2026-11-10,2026-11-09,VNT.AX,buy,1635,4.74474,7757.65,13.3788,155216.85


### Headline results

Every figure below is read straight from the artifacts the stages just
wrote (`data/reports/backtest_metrics.json`).

Note: these figures reflect the **current lake** — the incremental ingest
above refreshed and reconciled the most recent sessions, so the final
sessions (and therefore the closing NAV) can differ slightly from the
build documented in `docs/final_report.md`. That is the reconciliation
window doing its job, not a reproducibility failure: with the same bronze
input, every stage is deterministic.

In [9]:
m = artifacts.load_json_report("backtest_metrics")
print(f"NAV            : {m['initial_nav']:,.0f} -> {m['final_nav']:,.0f}  "
      f"({m['total_return']:+.2%})")
print(f"benchmark      : {m['benchmark_total_return']:+.2%}")
print(f"CAGR           : {m['cagr']:.2%} vs benchmark {m['benchmark_cagr']:.2%}")
print(f"Sharpe         : {m['sharpe_ratio']:.3f}")
print(f"max drawdown   : {m['max_drawdown']:.2%} "
      f"({m['max_drawdown_peak_date']} -> {m['max_drawdown_trough_date']})")
print(f"trading costs  : AUD {m['turnover']['total_costs_aud']:,.0f} "
      f"({m['turnover']['n_trades']:,} trades, avg turnover "
      f"{m['turnover']['avg_turnover_per_rebalance']:.1%})")

NAV            : 96,447 -> 156,933  (+62.71%)
benchmark      : +77.63%
CAGR           : 8.39% vs benchmark 9.98%
Sharpe         : 0.278
max drawdown   : -41.18% (2021-11-18 -> 2022-09-28)
trading costs  : AUD 94,474 (7,571 trades, avg turnover 56.5%)


### Refresh the platform snapshot

The `report` stage consolidates stage statuses, artifact freshness and
headline metrics into `data/reports/platform_summary.json` — the same
snapshot the dashboard reads.

In [10]:
rep = timed("report", stages.run_report_stage, settings)
status = artifacts.platform_status()
stages_status = pd.DataFrame(
    [{"stage": k, **v} if isinstance(v, dict) else {"stage": k, "status": v}
     for k, v in status.get("stages", {}).items()]
)
stages_status

2026-11-15 13:27:54 | INFO     | asx_quant.runs | Run started: 20261115T022754Z-d9a1539e


2026-11-15 13:27:54 | INFO     | asx_quant.runs | Stage started: report


2026-11-15 13:27:56 | INFO     | asx_quant.runs | Stage report: success


[   1.9s] report


,stage,status,reports
0,ingest,pass,"{'ingestion_summary': {'status': 'pass', 'warn..."
1,validate,pass,"{'data_quality': {'status': 'pass', 'warn_fail..."
2,transform,pass,"{'data_coverage': {'status': 'unknown', 'warn_..."
3,features,pass,"{'data_quality_features': {'status': 'pass', '..."
4,train,pass,"{'model_quality': {'status': 'pass', 'warn_fai..."
5,portfolio,pass,"{'portfolio_quality': {'status': 'pass', 'warn..."
6,backtest,pass,"{'backtest_quality': {'status': 'pass', 'warn_..."


### Keeping it current

Schedule the same chain without any notebook: `scripts/update.ps1` wraps
`uv run asx_quant update`, and `docs/automation.md` contains the Windows
Task Scheduler (`schtasks`) recipe for a weekly run matching the Monday
rebalance. For interactive monitoring, `uv run streamlit run
dashboard/Home.py` opens the read-only dashboard. Every stage is
deterministic and logged in `data/run_log.jsonl`, so a failed update is
repaired at the cause and simply re-run.